## Rag Pipeline: Data Ingestion to Vector DB Pipeline ##

In [1]:
import os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\Admin\AppData\Local\Temp\ipykernel_14592\329546744.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader
c:\Users\Admin\anaconda3\envs\rag-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
## Reading PDF files from a directory

def process_pdf_files(pdf_directory):
    pdf_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("*.pdf"))

    print(f"Found {len(pdf_files)} PDF files in the directory: {pdf_directory}")
    
    for pdf_file in pdf_files:
        loader = PyMuPDFLoader(str(pdf_file))
        documents = loader.load()

        #Add source information to metadata
        for doc in documents:
            doc.metadata["source_file"] = pdf_file.name
            doc.metadata["file_type"] = 'pdf'

        pdf_documents.extend(documents)

        print(f" Loaded {len(documents)} pages from {pdf_file.name}")
    return pdf_documents



all_pdf_documents = process_pdf_files("../data/pdf_files")

Found 2 PDF files in the directory: ../data/pdf_files
 Loaded 27 pages from 2026 Pinnacle Commissions_TBM AS KAM_GI PRIMA.pdf
 Loaded 1 pages from Aditya_Pagare_Resume.pdf


## Chunking  (Text splitting)

In [3]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200): 
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap,length_function=len, separators=["\n\n", "\n", " ", ""])
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")


    #show example of the first chunk
    if split_docs:
        print("Example of the first chunk:")
        print(split_docs[1].page_content)  # Print the first 500 characters of the first chunk
        print("Metadata:", split_docs[1].metadata)


    return split_docs


In [4]:
chunks = split_documents(all_pdf_documents, chunk_size=1000, chunk_overlap=200)

Split 28 documents into 53 chunks.
Example of the first chunk:
Proprietary and confidential — do not distribute
Dear Colleagues,
I want to begin by congratulating each one of you for your relentless commitment and determination in 
driving Abbott India Limited’s positive margins and growth throughout the year. Despite the challenges 
we faced, your perseverance and focus on Gross Margin Improvement have been truly commendable.
As we step into 2026, our collective priority must be consistent target achievement. To support you 
in this journey, we are excited to introduce the 2026 Pinnacle Commissions Program, designed to 
make earning opportunities simpler, more rewarding, and aligned with your efforts.
Key Highlights of 2026 Pinnacle Commissions Program:
1. Focus on strategic objectives: Components of monthly commission and league table now include 
strategic objectives for our focus brands and well as Business Units
2. Consistency is key: Quarter commissions will be basis consecutive 

## Embedding and VectorStoreDB

In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
class EmbeddingManager:

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):

        """Initialize the EmbeddingManager with a specified sentence transformer model.

        Args:
            model_name (str, optional): The name of the sentence transformer model to use for embeddings. Defaults to "all-MiniLM-L6-v2".
        """

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the sentence transformer model."""
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Loaded model: {self.model_name}, Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts.

        Args:
            texts (List[str]): A list of strings to generate embeddings for.
        """

        if not self.model:
            raise ValueError("Model is not loaded. Please check the model name or installation.")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## Initialize the EmbeddingManager and generate embeddings for the chunks
embedding_manager = EmbeddingManager(model_name="all-MiniLM-L6-v2")
embedding_manager

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3295.09it/s]


Loaded model: all-MiniLM-L6-v2, Embedding dimension: 384


## Vector Store using ChromaDB

In [7]:
class VectorStore:

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """Initialize the VectorStore with a specified collection name.

        Args:
            collection_name (str, optional): The name of the ChromaDB collection to use. Defaults to "pdf_documents".
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):

        """Initialize the ChromaDB client and collection."""
        try:
            #Create persistent chromadb client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #Get or create the collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Collection of PDF document embeddings for RAG project"}
                )
            
            print(f"Initialized ChromaDB collection: {self.collection_name} at {self.persist_directory}")
            print(f"Existing documents in the collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing ChromaDB collection: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add documents and their embeddings to the ChromaDB collection.

        Args:
            documents (List[Any]): A list of LangChain Document objects to add to the collection.
            embeddings (np.ndarray): A 2D numpy array of embeddings corresponding to the documents.
        """
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents must match the number of embeddings.")
        
        print(f"Adding {len(documents)} documents to the collection: {self.collection_name}")

        #Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_texts = []
        embeddings_list = []

        for i, (doc, emb) in enumerate(zip(documents, embeddings)):
            #Generate a unique ID for each document
            doc_id = str(uuid.uuid4())
            ids.append(doc_id)

            #Prepare metadata 
            metadata = dict(doc.metadata)  # Ensure metadata is a dictionary
            metadata["doc_index"] = i  # Add index to metadata
            metadata["context_length"] = len(doc.page_content)  # Add context length to metadata
            metadatas.append(metadata)
           
            #Store the document text and embedding
            documents_texts.append(doc.page_content)
            embeddings_list.append(emb)

        #Add to ChromaDB collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_texts
            )
            print(f"Added {len(documents)} documents to the collection: {self.collection_name}")
            print(f"Total documents in the collection after addition: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to ChromaDB collection: {e}")
            raise

vector_store = VectorStore()
vector_store


Initialized ChromaDB collection: pdf_documents at ../data/vector_store
Existing documents in the collection: 197


In [8]:
#Converting the text to embeddings

texts = [doc.page_content for doc in chunks]


#Generate embeddings for the chunks
embeddings = embedding_manager.generate_embeddings(texts)

#store the documents and embeddings in the vector store
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 53 texts...


Batches: 100%|██████████| 2/2 [00:05<00:00,  2.89s/it]


Generated embeddings with shape: (53, 384)
Adding 53 documents to the collection: pdf_documents
Added 53 documents to the collection: pdf_documents
Total documents in the collection after addition: 250


In [9]:

print(chunks[1].page_content)

Proprietary and confidential — do not distribute
Dear Colleagues,
I want to begin by congratulating each one of you for your relentless commitment and determination in 
driving Abbott India Limited’s positive margins and growth throughout the year. Despite the challenges 
we faced, your perseverance and focus on Gross Margin Improvement have been truly commendable.
As we step into 2026, our collective priority must be consistent target achievement. To support you 
in this journey, we are excited to introduce the 2026 Pinnacle Commissions Program, designed to 
make earning opportunities simpler, more rewarding, and aligned with your efforts.
Key Highlights of 2026 Pinnacle Commissions Program:
1. Focus on strategic objectives: Components of monthly commission and league table now include 
strategic objectives for our focus brands and well as Business Units
2. Consistency is key: Quarter commissions will be basis consecutive quarter target achievement


## Relevant Document Retrieval

In [14]:
class RAGRetriever:

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """Initialize the RAGRetriever with a specified VectorStore.
        This handles the retrieval of relevant documents based on a query using embeddings.

        Args:
            vector_store (VectorStore): An instance of the VectorStore class to retrieve documents from.
            embedding_manager (EmbeddingManager): An instance of the EmbeddingManager class to handle embeddings.
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """Retrieve the top_k most relevant documents for a given query.

        Args:
            query (str): The query string to search for relevant documents.
            top_k (int, optional): The number of top documents to retrieve. Defaults to 5.
            score_threshold (float, optional): The minimum similarity score for retrieved documents. Defaults to 0.0.   

        Returns:
            List[Dict[str, Any]]: A list of dictionaries containing the retrieved documents and their similarity scores.

        """
        print(f"Retrieving documents for query: '{query}' with top_k={top_k} and score_threshold={score_threshold}")
        print(f"Top k: {top_k}, Score threshold: {score_threshold}")

        #Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        #Retrieve all documents and their embeddings from the vector store
        try:
            results = self.vector_store.collection.query(   
                
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            #process results
            retrieved_docs = []


            if results and "documents" in results and results["documents"]:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]  # Assuming distances are returned in the results
                ids = results["ids"][0]  # Assuming ids are returned in the results

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance  # Convert distance to similarity score
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents after applying score threshold of {score_threshold}")
            else:
                print(f"No documents found")

            return retrieved_docs
            
        except Exception as e:
            print(f"An error occurred while retrieving documents: {e}")
            return []
        

rag_retriever = RAGRetriever(vector_store=vector_store, embedding_manager=embedding_manager)


In [20]:
z  = rag_retriever.retrieve("What are the highlights of the 2026 Pinnacle Commissions Program?", top_k=5, score_threshold=0.05)  


Retrieving documents for query: 'What are the highlights of the 2026 Pinnacle Commissions Program?' with top_k=5 and score_threshold=0.05
Top k: 5, Score threshold: 0.05
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 32.10it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents after applying score threshold of 0.05


#

# Augmented Generation

Integration Vectordb context pipeline with llm output

In [36]:
# Simple RAG pipeline
from urllib import response

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

#Initialize the Groq LLM
groq_api_key = os.getenv("GROQ_API_KEY")



llm = ChatGroq( groq_api_key = groq_api_key, model_name ="openai/gpt-oss-120b", temperature=0.1, max_tokens=1024)

#simple RAG pipeline function
def rag_simple(query, retriever, llm, top_k=3):

    #Retrieve relevant documents
    results = retriever.retrieve(query, top_k=top_k)

    context = "\n\n".join([doc["content"] for doc in results]) if results else "No relevant documents found."

    if not context:
            return "No relevant documents found for the query."

    #generate response using the LLM
    prompt = f"""Use the following context to answer the question concisely:
        Context:
        {context}
        Question: {query}

        Answer:"""
        
    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content


In [37]:
answer = rag_simple("What are the highlights of the 2026 Pinnacle Commissions Program?",rag_retriever,llm)
print(answer) 


Retrieving documents for query: 'What are the highlights of the 2026 Pinnacle Commissions Program?' with top_k=3 and score_threshold=0.0
Top k: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 42.29it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents after applying score threshold of 0.0


The 2026 Pinnacle Commissions Program highlights are:

1. **Focus on strategic objectives** – monthly commissions and league tables now incorporate strategic goals for focus brands and business units.  
2. **Consistency is key** – quarterly commissions are based on achieving targets in consecutive quarters.


In [ ]:
answer = rag_simple("what are the goals for 2026 commission?",rag_retriever,llm)
print(answer) 


Retrieving documents for query: 'what are the goals for 2026 commission?' with top_k=3 and score_threshold=0.0
Top k: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 36.34it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents after applying score threshold of 0.0


The 2026 commission program aims to :

- Align monthly commissions and league‑table rankings with the strategic objectives of Abbott’s focus brands and business units.  
- Reward consistent performance by basing quarterly commissions on achieving targets in consecutive quarters.


## Enhanced RAG pipeline features